In [176]:
import pandas as pd
import json
import numpy as np

In [ ]:
import xml.etree.ElementTree as ET
import json
import csv

def extract_cells(drawio_file, json_out="data.json"):
    tree = ET.parse(drawio_file)
    root = tree.getroot()

    extracted = []

    def clean_value(raw_value: str) -> str:
        if not raw_value:
            return ""
        if "<" in raw_value and ">" in raw_value:
            return BeautifulSoup(raw_value, "html.parser").get_text(strip=True)
        return raw_value.strip()


    for cell in root.iter("mxCell"):
        entry = {
            "id": cell.get("id"),
            "parent_id": cell.get("parent"),
            "value": clean_value(cell.get("value"))
        }
        if any(entry.values()):  # keep only if something is non-empty
            extracted.append(entry)

    # Save JSON
    with open(json_out, "w", encoding="utf-8") as f:
        json.dump(extracted, f, indent=2, ensure_ascii=False)

    print(f"✅ Extracted {len(extracted)} mxCell entries into {json_out}")


In [180]:

def nest_cells(json_file, output_file="nested.json"):
    with open(json_file, "r", encoding="utf-8") as f:
        cells = json.load(f)

    # Build lookup dictionary
    nodes = {cell["id"]: {**cell, "children": []} for cell in cells} # **cell unpacks all key-value pairs from the original dictionary
    roots = []

    # Assign children to their parent
    for cell in nodes.values():
        parent_id = cell.get("parent_id")
        if parent_id and parent_id in nodes:
            nodes[parent_id]["children"].append(cell)
        else:
            roots.append(cell)

    # Save nested JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(roots, f, indent=2, ensure_ascii=False)

    print(f"✅ Nested structure saved to {output_file}")



In [179]:
def flatten_columns(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # clean data, remove nodes that are unneccessary
    new_data = [node for node in data[0]["children"][0]["children"][2:] if node.get("children")]

    # pass these entities into a list
    list_entity = []
    for child in new_data:
        final = {}
        final["table"] = child["value"]
        columns = []

        """If a node has children with empty values, lift the grandchildren up""" 
        for grandchild in child["children"]:        
            if not grandchild.get("value") and grandchild.get("children"):
                for e in grandchild.get("children"):
                    if e.get("value") and e["value"] not in ("FK", "PK"):
                        columns.append(e["value"])

        final["columns"] = columns
        list_entity.append(final)


    # add error data
    w_insured_master_d = {
                            "table": "w_insured_master_d",
                            "columns": [
                                "w_insured_id",
                                "insured_src_id",
                                "insured_name",
                                "lob",
                                "insured_object",
                                "insured_description",
                                "w_integration key",
                                "w_valid_from",
                                "w_valid_to",
                                "w_valid_flag",
                                "w_created_time",
                                "w_updated_time",
                                "w_job_id"
                                ]
    }

    list_entity.append(w_insured_master_d)


    # return output to dataframe

    df = pd.DataFrame(list_entity)
    df_exploded = df.explode('columns').reset_index(drop=True)

    return df_exploded


In [181]:
def clean_data_before_export_csv():
    
    df = flatten_columns("nested.json")

    # add model entity status into file
    mask = df["table"].str.contains("WIP")
    df["model_status"] = np.where(mask, "TBU", "updated")

    # rename table by remove (WIP))
    df["table"] = df["table"].str.replace(r"\(WIP\)", "", regex=True).str.strip()

    # remove unneccessary characters

    df["table"] = df["table"].str.replace(r"&nbsp;&nbsp;", "", regex=True).str.strip()
    df["columns"] = df["columns"].str.replace(r"&nbsp;","", regex=True).str.strip()


    # remove line not entity
    df = df[~df["table"].str.contains("Policy Information", case=False, na=False)].reset_index(drop=True)

    return df

In [182]:
extract_cells("GIC_Model_v3.drawio")
nest_cells("data.json", output_file="nested.json")


df = flatten_columns("nested.json")
df = clean_data_before_export_csv()
df.to_csv('list_entity.csv', index=False)

✅ Extracted 2602 mxCell entries into data.json


FileNotFoundError: [Errno 2] No such file or directory: 'data.json'